In [ ]:
import torch
import torch.nn as nn
import torch.optim as opt
import numpy as np
import torch_npu
from torch.utils.data import DataLoader, Dataset
from models.unet_v1 import UNet
from models.ddpm import DDPM
# from models.simplenet import UNet
# from models.ddpm_nocond import DDPM_nocond as DDPM
import os
from datetime import datetime
import pandas as pd
import numpy as np
num_steps = 300
repaint_steps = 10
jump_len = 10
N = 10
n_samples = 1

device = "npu:0"
model_save_path = "/home/docker/code/Aurora_DDPM_final/ckpt/cond/ckptv3_unetv1/aurora_diff_best.pth"
unet = UNet(1, 1)
ddpm = DDPM(unet, num_train_steps=1000)
checkpoint = torch.load(model_save_path, map_location=device)
ddpm.load_state_dict(checkpoint['model_state_dict'],strict=False)
#ddpm.load_state_dict(checkpoint) 

<All keys matched successfully>

In [29]:
from datetime import datetime
import pickle
import pandas as pd
import numpy as np
from data.dataset_diff2 import load_all_aurora_data, OmniDataset
# 训练数据
data_mn_all,_ = load_all_aurora_data(years=[1996], months=range(1, 3))

mn_mean = data_mn_all.mean()
mn_std = data_mn_all.std()
mn_var = data_mn_all.var()
mn_max = data_mn_all.max()
mn_min = data_mn_all.min()

# 测试数据
polar_data_all = np.load("/home/docker/data/private/AuroraData/real_aurora_data_polar/1996/resampled_5min_1996_0405.npy",allow_pickle=True)
polar_timestamps = polar_data_all['utc']
polar_data = np.stack(polar_data_all['aurora_image'], axis=0).astype(np.float32)
polar_mean = polar_data.mean()
polar_std = polar_data.std()
polar_var = polar_data.var()
polar_max = polar_data.max()
polar_min = polar_data.min()

# polar -> ovation
#polar_data[polar_data <= 0] = 0
#polar_data_transfer = ((polar_data - polar_mean) /polar_std) * mn_std + mn_mean
polar_data_transfer = (polar_data - polar_min) / (polar_max - polar_min) * (mn_max - mn_min) + mn_min
polar_tran_mean = polar_data_transfer.mean()
polar_tran_std = polar_data_transfer.std()
polar_tran_var = polar_data_transfer.var()
polar_tran_max = polar_data_transfer.max()
polar_tran_min = polar_data_transfer.min()

加载极光数据: aurora_img_19960101.npy, 形状: (8928, 80, 96)
加载OMNI数据: omni_19960101_5min.npy, 形状: (8928, 5)
加载极光数据: aurora_img_19960201.npy, 形状: (8352, 80, 96)
加载OMNI数据: omni_19960201_5min.npy, 形状: (8352, 5)
极光数据总形状: (17280, 80, 96)
OMNI数据总形状: (17280, 5)
对齐后极光数据形状: (17280, 80, 96)
对齐后OMNI数据形状: (17280, 5)


In [30]:
# 获取polar数据对应时刻的mn数据和solar wind数据
data1_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19960401.npy"
data2_path = "/home/docker/data/private/AuroraData/generated_aurora_data/1996_omni_aurora/aurora_img_19960501.npy"
mn_data1 = np.load(data1_path)
mn_data2 = np.load(data2_path)
mn_data = np.concatenate((mn_data1, mn_data2), axis=0)
omni_path1 = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min/1996/omni_19960401_5min.npy"
omni_path2 = "/home/docker/data/private/AuroraData/omni_real_data/omni_5min/1996/omni_19960501_5min.npy"
omni_data1 = np.load(omni_path1)
omni_data2 = np.load(omni_path2)
omni_data = np.concatenate((omni_data1, omni_data2), axis=0)
mn_time = omni_data['utc']

solar_fields = ['Bx', 'By', 'Bz', 'V', 'P']
solar_components = []
for field in solar_fields:
    field_data = omni_data[field]
    if field_data.ndim > 1:
        if field_data.shape[1] > 0:
            field_data = field_data[:, 0]
        else:
            field_data = field_data.flatten()
    solar_components.append(field_data.astype(np.float32))
solar_data = np.column_stack(solar_components)
solar_data = OmniDataset(solar_data)

In [23]:
print("mn_mean:", mn_mean)
print("mn_std:", mn_std)
print("mn_var:", mn_var)
print("mn_max:", mn_max)
print("mn_min:", mn_min)

mn_mean: 0.26400631244055567
mn_std: 0.5349329291722651
mn_var: 0.28615323871281956
mn_max: 8.411953842459841
mn_min: 0.0


In [18]:
print("polar_mean:", polar_mean)
print("polar_std:", polar_std)
print("polar_var:", polar_var)
print("polar_max:", polar_max)
print("polar_min:", polar_min)

polar_mean: 0.5087137
polar_std: 2.3720517
polar_var: 5.6266294
polar_max: 39.999454
polar_min: 0.0


In [6]:
print("polar_tran_mean:", polar_tran_mean)
print("polar_tran_std:", polar_tran_std)
print("polar_tran_var:", polar_tran_var)
print("polar_tran_max:", polar_tran_max)
print("polar_tran_min:", polar_tran_min)

polar_tran_mean: 0.10698368
polar_tran_std: 0.49884763
polar_tran_var: 0.24884896
polar_tran_max: 8.411954
polar_tran_min: 0.0


In [31]:
class normalize(nn.Module):
    def __init__(self, datas):
        super().__init__()
        self.datas = datas
        log_data = np.log1p(self.datas)
        self.log_min = float(log_data.min())
        self.log_max = float(log_data.max())
        self.log_range = max(self.log_max - self.log_min, 1e-6)
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        x = np.log1p(x)
        x = (x - self.log_min) / self.log_range
        x = np.clip(x, 0.0, 1.0)
        # return (x * 2.0 - 1.0).astype(np.float32)
        return x.astype(np.float32)
    
class denormalize(nn.Module):
    def __init__(self, datas):
        super().__init__()
        self.datas = np.array(datas, dtype=np.float32) 
        log_data = np.log1p(self.datas)
        self.log_min = float(log_data.min())
        self.log_max = float(log_data.max())
        self.log_range = max(self.log_max - self.log_min, 1e-6)
        
    def forward(self, x: np.ndarray) -> np.ndarray:
        # aurora_01 = (x + 1.0) / 2.0
        aurora_01 = x
        aurora_log = aurora_01 * self.log_range + self.log_min
        aurora_flux = np.expm1(aurora_log)
        return aurora_flux
normalizer_trans = normalize(polar_data_transfer)
denormalizer_trans = denormalize(polar_data_transfer)

normalizer_real = normalize(polar_data)
denormalizer_real = denormalize(polar_data)

normalizer_all = normalize(data_mn_all)
denormalizer_all = denormalize(data_mn_all)



In [37]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.image import imread
import cartopy.feature as carfeat
import cartopy.io.shapereader as shpreader
from matplotlib.colors import LinearSegmentedColormap
from cartopy.feature.nightshade import Nightshade
import scipy.ndimage
import scipy.interpolate
from datetime import datetime
import os
import math
import aacgmv2
from datetime import timedelta
def convert_datetime64_to_datetime(dt64):
    """将 numpy.datetime64 转换为 datetime.datetime"""
    if dt64 is None:
        return None
    if isinstance(dt64, datetime):
        return dt64
    import pandas as pd
    return pd.Timestamp(dt64).to_pydatetime()

def plot_real(timestamp, energy_flux, save_path):
    # 1. 准备你的数据
    # 假设你有网格数据：energy_flux[mlat_bins, mlt_bins]
    timestamp = convert_datetime64_to_datetime(timestamp)
    mlat = np.linspace(50, 90, 80)  # 纬度网格
    mlt = np.linspace(0, 24, 96)    # 地方时网格
    MLAT, MLT = np.meshgrid(mlat, mlt)  # 创建网格

    # 2. 转换为绘图坐标（极坐标）
    # 将MLT转换为角度（弧度）
    # 减去π/2使0 MLT在底部（-90度）
    theta = (MLT / 24.0) * 2 * np.pi - np.pi/2

    # 将MLAT转换为半径
    # 纬度越高（接近90°），半径越小
    # 我们想要从中心（90°）到边缘（50°）的半径从0到1
    r = (90 - MLAT) / 40.0  # 因为90-50=40度范围

    x = r * np.cos(theta)
    y = r * np.sin(theta)

    # 3. 创建极坐标投影的地图
    # fig = plt.figure(figsize=(8, 8))
    # # 使用PlateCarree投影的极坐标视图
    # ax = plt.axes(projection=ccrs.NorthPolarStereo())
    # ax.set_extent([-180, 180, 50, 90], crs=ccrs.PlateCarree())

    fig, ax = plt.subplots(figsize=(6, 5), subplot_kw={'projection': 'polar'})

    # 4. 绘制填色图
    # 4. 自定义颜色映射 - 黑->蓝->绿->红->灰
    # 这是论文中常用的色彩映射
    colors = [
        (0, 0, 0),          # 黑色 (最低值)
        (0, 0, 0.5),        # 深蓝
        (0, 0, 0.8),        # 蓝色
        (0, 0.5, 1),        # 天蓝
        (0, 1, 1),          # 青色
        (0.5, 1, 0.5),      # 青绿
        (1, 1, 0),          # 黄色
        (1, 0.5, 0),        # 橙色
        (1, 0, 0),          # 红色
        (0.8, 0.8, 0.8)     # 灰色 (最高值)
    ]
    custom_cmap = LinearSegmentedColormap.from_list('aurora_cmap', colors, N=256)


    # ==================== 5. 设置极坐标图的属性 ====================
    ax.set_theta_zero_location('S')  # 0度在底部（对应0 MLT）
    ax.set_theta_direction(1)       # 角度顺时针增加（这是空间物理的标准）
    ax.set_ylim(0, 1)                # 半径范围从0到1

    # 隐藏默认的径向标签（我们要用纬度标签）
    ax.set_yticklabels([])

    # 设置角度刻度为地方时
    hour_ticks = np.arange(0, 24, 1)
    angle_ticks = (hour_ticks / 24.0) * 360  # 转换为角度
    ax.set_xticks(np.deg2rad(angle_ticks))

    # 设置刻度标签 - 只显示0,6,12,18，其他为空
    mlt_labels = []
    for hour in hour_ticks:
        if hour in [0, 6, 12, 18]:
            mlt_labels.append(str(hour))
        else:
            mlt_labels.append('')
    ax.set_xticklabels(mlt_labels, fontsize=10)

    # 7. 设置8个径向网格线（纬度圈）的位置
    # 从50°到85°，每5°一个，共8个圈
    lat_circles = [50, 55, 60, 65, 70, 75, 80, 85]
    radial_ticks = [(90 - lat) / 40.0 for lat in lat_circles]  # 转换为半径

    # 设置径向网格线的位置和标签
    ax.set_rticks(radial_ticks)
    ax.set_yticklabels([f'{lat}°' for lat in lat_circles], 
                    fontsize=5, color='white')
    time_str = timestamp.strftime("%Y-%m-%d %H:%M UT")
    c = ax.pcolormesh(theta, r, energy_flux.T,
                    cmap=custom_cmap, shading='auto',vmin=0, vmax=5)
    title = "Auroral Energy Flux Map"
    fig.text(0.5, 0.95, f'{title} - {time_str}',
                color='black', fontsize=8, ha='center', weight='bold')
    cbar_ax = fig.add_axes([0.85, 0.25, 0.03, 0.5])  # [left, bottom, width, height]
    plt.colorbar(c, cax=cbar_ax, pad=0.1, label='ergs cm⁻² s⁻¹')

    plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]
    plt.savefig(save_path, dpi=300, facecolor='white')
    #plt.show()

def plot(timestamp, repaired_flux, save_path):
    timestamp = convert_datetime64_to_datetime(timestamp)
    lat_coords = np.linspace(50, 90, 80)
    mlt_coords = np.linspace(0.0, 24.0, 96)
    mltN, mlatN = np.meshgrid(mlt_coords, lat_coords)
    mlonN_1D_small = aacgmv2.convert_mlt(mltN[0], timestamp, m2a=True)
    mlonN_1D = np.tile(mlonN_1D_small, mlatN.shape[0])
    mlatN_1D = np.squeeze(mlatN.reshape(np.size(mltN), 1))

    (glatN_1D, glonN_1D, galtN) = aacgmv2.convert_latlon_arr(mlatN_1D, mlonN_1D, 100, timestamp,
                                                                        method_code="A2G")

    # 插值到世界地图网格
    geo_2D = np.vstack((glatN_1D, glonN_1D)).T
    fluxN_1D = repaired_flux.reshape(7680, 1)

    # 创建世界地图网格 - 使用更小的网格
    h, w = 512, 1024
    wx, wy = np.mgrid[-90:90:180 / h, -180:180:360 / w]

    # 线性插值
    aimg = np.squeeze(scipy.interpolate.griddata(geo_2D, fluxN_1D, (wx, wy), method='linear', fill_value=0))

    # 高斯平滑处理
    aimg = scipy.ndimage.gaussian_filter(aimg, sigma=(2, 3), mode='wrap')

    aimg = aimg.astype(np.float32)

    colors = [
            (0.0, 0.0, 0.0),
            (0.0, 0.2, 0.0),
            (0.0, 0.5, 0.0),
            (0.0, 0.8, 0.0),
            (0.5, 1.0, 0.0),
            (1.0, 1.0, 0.0),
            (1.0, 0.6, 0.0),
            (1.0, 0.3, 0.0),
            (1.0, 0.0, 0.0),
        ]
    cmap = LinearSegmentedColormap.from_list('aurora', colors, N=256)
    fig = plt.figure(figsize=(12, 10), dpi=150)
    fig.set_facecolor('white')

    ax = fig.add_subplot(1, 1, 1,
                            projection=ccrs.Orthographic(116.2,90))
                            #position=[0, 0, 1, 1])
                            #position=[0.3, 0.1, 0.45, 0.45])
    
    background_img = '/home/docker/data/private/AuroraData/background_img/natural-earth-1_large2048px.png'
    map_img = imread(background_img)
    ax.imshow(map_img, origin='upper', transform=ccrs.PlateCarree(),
                extent=[-180, 180, -90, 90], zorder=0)


    gl = ax.gridlines(linestyle='solid', alpha=0.5, color='white')
    gl.n_steps = 100
    gl.xlocator = matplotlib.ticker.FixedLocator(np.arange(-180, 190, 45))
    gl.ylocator = matplotlib.ticker.FixedLocator(np.arange(-90, 100, 10))



    ax.coastlines('10m', color='white', alpha=0.4)

    ax.add_feature(Nightshade(timestamp))
    img = ax.imshow(aimg,
                    # vmin=0,
                    # vmax=5,
                    transform=ccrs.PlateCarree(),
                    extent=[-180, 180, -90, 90],
                    origin='lower',
                    zorder=3,
                    alpha=0.8,
                    cmap=cmap)
    ax.set_facecolor('white')

    if timestamp is not None:
        time_str = timestamp.strftime("%Y-%m-%d %H:%M UT")
    else:
        time_str = "Unknown Time"
    title = "Aurora ForecastNet Repaired Aurora"
    fig.text(0.5, 0.95, f'{title} - {time_str}',
                color='white', fontsize=18, ha='center', weight='bold')

    cbaxes = fig.add_axes([0.3, 0.07, 0.4, 0.02])
    cbar = plt.colorbar(img, cax=cbaxes, orientation='horizontal')
    cbar.set_alpha(1)
    cbar.ax.tick_params(labelsize=15, colors='white')
    cbar.set_label(r'aurora flux $\mathrm{erg\/cm^{-2}\/s^{-1}}$',
                    color='white', fontsize=16)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, facecolor='black')



In [8]:
def create_mask_in_mlt_region(image_shape, mlat_range=(65, 75), mlt_range = (1, 3)):
    """
    在指定的MLT区域创建掩码
    
    Args:
        image_shape: 图像形状 (80, 96)
        mlat_range: 纬度范围，默认为65-75度
    
    Returns:
        mask: 掩码矩阵，1表示保留，0表示挖去
    """
    h, w = image_shape
    # MLT转换为列索引 (96列对应0-24小时)
    # 每列对应的MLT小时：col_idx * 24 / 96
    
    mlt_start, mlt_end = mlt_range
    
    
    col_start2 = int(mlt_start * w / 24)
    col_end2 = int(mlt_end * w / 24)
    
    
    x_start2 = col_start2
    x_end2 = col_end2
    
    mlat_min, mlat_max = mlat_range
    
    # 计算行索引范围
    row_min = int((90 - mlat_max) * h / 40)  # 30
    row_max = int((90 - mlat_min) * h / 40)  # 50
    
    y_start = row_min
    y_end = row_max
    
    # 创建掩码
    mask = np.ones(image_shape)
    mask[y_start:y_end, x_start2:x_end2] = 0
    
    return mask

In [ ]:
mask1 = create_mask_in_mlt_region(
            (80,96), 
            mlat_range=(70, 75),
            mlt_range=(1, 3) )
mask2 = create_mask_in_mlt_region(
            (80,96),
            mlat_range=(68, 77),
            mlt_range=(0.75, 3.25) )

mask2[mask1 == 0] = 1

In [46]:
### 
time1 = polar_timestamps[0].astype('datetime64[s]')
time1 = pd.Timestamp(time1).to_pydatetime()
base_time = datetime.fromisoformat("1996-04-01T00:00:00")
delta = time1 - base_time
idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
mn_data = data_mn_all[idx_mn]

print("polar time:",time1)
print("mn_time:", mn_time[idx_mn])
solar_point = solar_data[idx_mn]
solar_point = solar_point.unsqueeze(0).to(device)
# polar数据
polar_transfer = polar_data_transfer[0]

# mask

mask = create_mask_in_mlt_region(
            (80,96), 
            mlat_range=(70, 75),
            mlt_range=(1, 3) )

# mask = np.ones_like(polar_transfer)
# real_close_to_zero = polar_transfer == 0
# sim_has_value = mn_data > 0
# need_repair = real_close_to_zero & sim_has_value
# mask[need_repair] = 0.0

mask = np.expand_dims(mask, axis=(0, 1))
mask = torch.tensor(mask).float().to(device)

polar_transfer = normalizer_all(polar_transfer)
polar_transfer = np.expand_dims(polar_transfer, axis=(0,1))
polar = torch.tensor(polar_transfer).float().to(device)

ddpm.eval()
ddpm.to(device)
with torch.no_grad():
    # 使用ddpm.sample进行修补
    repaired = ddpm.sample(
        polar,
        mask,
        solar_point,
        num_inference_steps=num_steps,
        n_sample=n_samples,
        j=jump_len,
        r=repaint_steps,
    )
repaired_np = repaired.cpu().numpy().squeeze()
repaired_flux = denormalizer_all(repaired_np)

polar time: 1996-04-01 00:00:00
mn_time: 1996-04-01T00:00:00.000000000


In [47]:
mask = mask.cpu().numpy().squeeze()
# polar_data_transfer = (polar_data - polar_min) / (polar_max - polar_min) * (mn_max - mn_min) + mn_min
# repaired_flux_re = (repaired_flux - mn_mean) / mn_std * polar_std + polar_mean
repaired_flux_re = (repaired_flux - mn_min) / (mn_max - mn_min) * (polar_max - polar_min) + polar_min
repaired_flux_mask = repaired_flux_re * (1-mask)
#mask = mask.cpu().numpy().squeeze()
# repaired_flux_mask = repaired_flux * (1-mask)
# repaired_flux_re = (repaired_flux_mask - mn_mean) / mn_std * polar_std + polar_mean

In [25]:
real_flux = polar_data[0]
real_flux_mask = polar_data[0] * (1-mask)

mn_data = data_mn_all[idx_mn]
mn_data_mask = mn_data * (1-mask)

In [48]:
time1 = polar_timestamps[0].astype('datetime64[s]')
time1 = pd.Timestamp(time1).to_pydatetime()
base_time = datetime.fromisoformat("1996-04-01T00:00:00")
delta = time1 - base_time
idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
mn_data = data_mn_all[idx_mn]

print("polar time:",time1)
print("mn_time:", mn_time[idx_mn])
solar_point = solar_data[idx_mn]
solar_point = solar_point.unsqueeze(0).to(device)
polar = polar_data[0]
# mask
# mask = create_mask_in_mlt_region(
#             (80,96), 
#             mlat_range=(70, 75),
#             mlt_range=(1, 3))
mask = np.ones_like(polar)
real_close_to_zero = polar == 0
sim_has_value = mn_data > 0
need_repair = real_close_to_zero & sim_has_value
mask[need_repair] = 0.0
mask = np.expand_dims(mask, axis=(0, 1))
mask = torch.tensor(mask).float().to(device)
#mask = np.expand_dims(mask, axis=(0, 1))


# mask2 = create_mask_in_mlt_region(
#             (80,96),
#             mlat_range=(68, 77),
#             mlt_range=(0.75, 3.25) 
#mask2[mask == 0] = 1


# polar数据

#polar[mask2==0] = polar[mask2==0]/3

#polar = normalizer_real(polar)
polar = (polar - polar_min) / (polar_max - polar_min) * (mn_max - mn_min) + mn_min
polar = normalizer_all(polar)
polar = np.expand_dims(polar, axis=(0,1))
polar = torch.tensor(polar).float().to(device)

ddpm.eval()
ddpm.to(device)
with torch.no_grad():
    # 使用ddpm.sample进行修补
    repaired = ddpm.sample(
        polar,
        mask,
        solar_point,
        num_inference_steps=num_steps,
        n_sample=n_samples,
        j=jump_len,
        r=repaint_steps,
    )
repaired_np = repaired.cpu().numpy().squeeze()
#repaired_flux = denormalizer_real(repaired_np)

repaired_flux = denormalizer_all(repaired_np)
#repaired_flux = denormalizer_real(repaired_np)
repaired_flux = (repaired_flux - mn_min) / (mn_max - mn_min) * (polar_max - polar_min) + polar_min
# plot_real(time1, repaired_flux, "re_polar_2.png")
# repaired_flux = (repaired_flux - mn_mean) / mn_std * polar_std + polar_mean
# plot_real(time1, repaired_flux, "re_polar_3.png")

polar time: 1996-04-01 00:00:00
mn_time: 1996-04-01T00:00:00.000000000


In [49]:
mask = mask.cpu().numpy().squeeze()
repaired_flux_mask_2 = repaired_flux * (1-mask)
repaired_flux_mask_3 = repaired_flux * mask
#repaired_flux_mask_2[mask == 0] = repaired_flux_mask_2[mask == 0]*2

In [ ]:
# 将polar_data标准化到与mn_data_all相同的均值和标准差
polar_data_transfer = ((polar_data - polar_mean) /polar_std) * mn_std + mn_mean
polar_transfer = polar_data_transfer[0]

polar= polar_data[0]
time1 = polar_timestamps[0].astype('datetime64[s]')
time1 = pd.Timestamp(time1).to_pydatetime()
#plot_real(time1, polar, "real_polar.png")


    
normalizer_real = normalize(polar_data)
denormalizer_real = denormalize(polar_data)

normalizer_trans = normalize(polar_data_transfer)
denormalizer_trans = denormalize(polar_data_transfer)

# 确定polar数据对应的模拟数据
base_time = datetime.fromisoformat("1996-04-01T00:00:00")
delta = time1 - base_time
idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
mn_data = data_mn_all[idx_mn]
print("polar time:",time1)
print("mn_time:", mn_time[idx_mn])

# mask数据
mask = np.ones_like(polar)
real_close_to_zero = polar == 0
sim_has_value = mn_data > 0
need_repair = real_close_to_zero & sim_has_value
mask[need_repair] = 0.0
plot_real(time1, mask, "mask.png")

mask = np.expand_dims(mask, axis=(0, 1))

# polar数据
#polar = normalizer_real(polar)
polar = (polar - polar_min) / (polar_max - polar_min) * (mn_max - mn_min) + mn_min
polar = normalizer_all(polar)
polar = np.expand_dims(polar, axis=(0,1))

# 转换成tensor
mask = torch.tensor(mask).float().to(device)
polar = torch.tensor(polar).float().to(device)

print("mask shape:", mask.shape)
print("polar shape:", polar.shape)

ddpm.eval()
ddpm.to(device)
with torch.no_grad():
    # 使用ddpm.sample进行修补
    repaired = ddpm.sample(
        polar,
        mask,
        num_inference_steps=num_steps,
        n_sample=n_samples,
        j=jump_len,
        r=repaint_steps,
    )
repaired_np = repaired.cpu().numpy().squeeze()
repaired_flux = denormalizer_real(repaired_np)
plot_real(time1, repaired_flux, "re_polar_2.png")
repaired_flux = (repaired_flux - mn_mean) / mn_std * polar_std + polar_mean
plot_real(time1, repaired_flux, "re_polar_3.png")

polar time: 1996-04-01 00:00:00
mn_time: 1996-04-01T00:00:00.000000000


/tmp/ipykernel_4135507/2673509482.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]


mask shape: torch.Size([1, 1, 80, 96])
polar shape: torch.Size([1, 1, 80, 96])


/tmp/ipykernel_4135507/2673509482.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]
/tmp/ipykernel_4135507/2673509482.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]


In [ ]:
def creat_gif(image_dir, duration=500, loop=0, image_nums=20):
    from PIL import Image
    pil_images = []
    for i in range(image_nums):
        img_path = os.path.join(image_dir, f'repaired_aurora_{i+1:02d}.png')
        pil_img = Image.open(img_path).copy()
        pil_images.append(pil_img)
    save_path = os.path.join(image_dir, 'repaired_aurora_animation.gif')
    pil_images[0].save(save_path, save_all=True, append_images=pil_images[1:],
                      duration=duration, loop=loop, optimize=False)

In [43]:
import scipy.ndimage as ndimage
def train(input_data, mask, solar_point ,time1, img_path):
    ddpm.eval()
    ddpm.to(device)
    with torch.no_grad():
        # 使用ddpm.sample进行修补
        repaired = ddpm.sample(
            input_data,
            mask,
            solar_point,
            num_inference_steps=num_steps,
            n_sample=n_samples,
            j=jump_len,
            r=repaint_steps,
        )
    repaired_np = repaired.cpu().numpy().squeeze()
    repaired_flux = denormalizer_all(repaired_np)
    #repaired_flux = denormalizer_real(repaired_np)
    repaired_flux = (repaired_flux - mn_min) / (mn_max - mn_min) * (polar_max - polar_min) + polar_min
    
    mask = mask.cpu().numpy().squeeze()
    input_data_np = input_data.cpu().numpy().squeeze()
    input_data_denorm = denormalizer_all(input_data_np)
    input_data_denorm = (input_data_denorm - mn_min) / (mn_max - mn_min) * (polar_max - polar_min) + polar_min
    
    soft_mask = ndimage.gaussian_filter(mask.astype(np.float32), sigma=2.0)
    soft_mask = np.clip(soft_mask, 0, 1)
    blended = (soft_mask * input_data_denorm) + ((1 - soft_mask) * repaired_flux)
    
    plot_real(time1, blended, img_path)
    
    #repaired_flux = (repaired_flux - mn_mean) / mn_std * polar_std + polar_mean
    #plot_real(time1, repaired_flux, img_path.replace("repaired_aurora", "re_aurora"),)
    #return repaired_flux

In [ ]:
polar_dir = "/home/docker/code/Aurora_DDPM_final/res/polar/repaired_ckptv0_unetv3"
os.makedirs(polar_dir, exist_ok=True)

normalizer_polar = normalize(polar_data)
time1 = polar_timestamps[170]
polar= polar_data[170]
time1 = polar_timestamps[170].astype('datetime64[s]')
time1 = pd.Timestamp(time1).to_pydatetime()
base_time = datetime.fromisoformat("1996-04-01T00:00:00")
delta = time1 - base_time
idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
mn_data = data_mn_all[idx_mn]

print("polar time:",time1)
print("mn_time:", mn_time[idx_mn])

solar_point = solar_data[170]
mask = np.ones_like(polar)
real_close_to_zero = polar ==0
mask[real_close_to_zero] = 0.0
mask = np.expand_dims(mask, axis=(0, 1))   
    
polar = normalizer_polar(polar)
polar = np.expand_dims(polar, axis=(0,1))
solar_point = solar_point.unsqueeze(0).to(device)
input_data = polar.copy()
input_data = torch.tensor(input_data).float().to(device)
mask = torch.tensor(mask).float().to(device)

img_path = os.path.join(polar_dir, f'repaired_aurora_{170}.png')
train(input_data, mask, solar_point,time1, img_path)

polar time: 1996-04-03 02:15:00
mn_time: 1996-04-03T02:15:00.000000000


/tmp/ipykernel_3187054/3499050754.py:208: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [44]:
polar_dir = "/home/docker/code/Aurora_DDPM_final/res/polar/repaired_ckptv1_unetv1/res3"
os.makedirs(polar_dir, exist_ok=True)

data_dict = {'utc': [], 'image': []}
for i in range(0, 15):
    time1 = polar_timestamps[i]
    polar= polar_data[i]
    time1 = polar_timestamps[i].astype('datetime64[s]')
    time1 = pd.Timestamp(time1).to_pydatetime()
    base_time = datetime.fromisoformat("1996-04-01T00:00:00")
    delta = time1 - base_time
    idx_mn = int(delta.total_seconds() // 300) # Assuming 5-minute intervals
    mn_data = data_mn_all[idx_mn]
    
    print("polar time:",time1)
    print("mn_time:", mn_time[idx_mn])
    
    solar_point = solar_data[idx_mn+1]
    mask = np.ones_like(polar)
    real_close_to_zero = polar == 0
    sim_has_value = mn_data > 0
    need_repair = real_close_to_zero & sim_has_value
    mask[need_repair] = 0.0
    mask = np.expand_dims(mask, axis=(0, 1))
       
    polar = (polar - polar_min) / (polar_max - polar_min) * (mn_max - mn_min) + mn_min
    polar = normalizer_all(polar)
    polar = np.expand_dims(polar, axis=(0,1))
    solar_point = solar_point.unsqueeze(0).to(device)
    input_data = polar.copy()
    input_data = torch.tensor(input_data).float().to(device)
    mask = torch.tensor(mask).float().to(device)
    
    img_path = os.path.join(polar_dir, f'repaired_aurora_{i}.png')
    train(input_data, mask, solar_point,time1, img_path)
    # repaired_flux = train(input_data, mask, solar_point,time1, img_path)
    # data_dict['utc'].append(time1)
    # data_dict['image'].append(repaired_flux)
    
# df = pd.DataFrame(
#     {
#         'utc': data_dict['utc'],
#         'image': data_dict['image']
#     }
# )
# save_path ="/home/docker/code/Aurora_DDPM/reasult/polar_res/new_res"
# df['utc'] = pd.to_datetime(df['utc'])
# df.sort_values(by='utc', inplace=True)
# df.reset_index(drop=True, inplace=True)
# structured_array = df.to_records(index=False)
# np.save(os.path.join(save_path, 'repaired_polar_unetV3.npy'), structured_array)

polar time: 1996-04-01 00:00:00
mn_time: 1996-04-01T00:00:00.000000000


/tmp/ipykernel_901317/1104974613.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]


polar time: 1996-04-01 00:05:00
mn_time: 1996-04-01T00:05:00.000000000


/tmp/ipykernel_901317/1104974613.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]


polar time: 1996-04-01 00:10:00
mn_time: 1996-04-01T00:10:00.000000000


...

polar time: 1996-04-01 00:45:00
mn_time: 1996-04-01T00:45:00.000000000


/tmp/ipykernel_901317/1104974613.py:114: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # [left, bottom, right, top]


polar time: 1996-04-01 00:50:00
mn_time: 1996-04-01T00:50:00.000000000


Process ForkServerPoolWorker-34:
Process ForkServerPoolWorker-30:
Process ForkServerPoolWorker-33:
Process ForkServerPoolWorker-29:
Process ForkServerPoolWorker-27:
Process ForkServerPoolWorker-31:
Process ForkServerPoolWorker-28:
Process ForkServerPoolWorker-32:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/docker/miniconda3/envs/torch/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/docker/miniconda3/envs/torch/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/docker/miniconda3/envs/torch/lib/python3.9/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/home/docker/miniconda3/envs/torch/lib/python3.9/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/docker/miniconda3/envs/

KeyboardInterrupt: 

In [ ]:
data1 = aurora_data[0]
#data2 = mn_data_all[0]
save_path_1 = "/home/docker/code/AuroraForecastNet_v1/aurora_1.png"
save_path_2 = "/home/docker/code/AuroraForecastNet_v1/repaired_aurora_17.png"
save_path_3 = "/home/docker/code/AuroraForecastNet_v1/mn_aurora_1.png" 
input_data1 = input_data.cpu().numpy().squeeze()
#plot(timestamp, data1, save_path_1)
plot(timestamps[0], repaired_flux, save_path_2)
# plot(timestamp, repaired_flux_2, save_path_2)